# Feature hashing: the hashing trick for huge category spaces

_Feature Engineering (Zheng & Casari)_

**Hash millions of categories into a fixed number of buckets — no vocabulary stored, no matter how many categories appear.**

Real data is full of categorical features with a gigantic number of possible values: every
       distinct word in a corpus, every user ID, every advertiser ID, every IP address. The usual way to feed
       a category to a model is one-hot encoding &mdash; one column per distinct value, with a single 1
       marking which value occurred. That is fine for a handful of categories. It is hopeless when there are
       millions: the feature vector becomes millions of columns wide, and you must store a giant
       vocabulary (a lookup table mapping each value to its column).

       Feature hashing &mdash; the "hashing trick" &mdash; sidesteps both problems. Fix a
       number of output buckets $m$ ahead of time (say a million). For each category, run it through a
       hash function $h$ that returns a bucket index between $0$ and $m-1$, and add $1$ to that bucket.
       The output is always an $m$-dimensional vector, no matter how many distinct categories exist,
       and you never store a vocabulary &mdash; the hash function is the mapping. This is Chapter 5 of
       Zheng & Casari's Feature Engineering for Machine Learning, and their running example is the
       Yelp reviews text.

---

This notebook is a practice scaffold for the **Feature hashing: the hashing trick for huge category spaces** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — scikit-learn

In [ ]:
import pandas as pd
import json
from sklearn.feature_extraction import FeatureHasher

# --- Load the Yelp reviews (Yelp Dataset Challenge, round 6) ---
# Get it from the book's repo: github.com/alicezheng/feature-engineering-book
f = open('yelp_academic_dataset_review.json')
review_df = pd.DataFrame([json.loads(x) for x in f.readlines()])
f.close()

# Each review's "category" tokens are simply its words.
# (For a true bag-of-words you'd tokenize; the book hashes the word list directly.)
def word_list(text):
    return text.lower().split()

categories = (word_list(t) for t in review_df['text'])

# === Feature hashing ===
# Fixed output size m = 2**18 buckets, NO vocabulary is built or stored.
h = FeatureHasher(n_features=2 ** 18, input_type='string')
f = h.transform(categories)        # sparse (n_reviews x 262144) matrix

print('hashed shape :', f.shape)   # -> (n_reviews, 262144), independent of vocab size

# === Storage comparison vs one-hot / bag-of-words ===
# The hashed matrix is sparse; measure how many bytes it actually occupies.
from sys import getsizeof
print('Our pandas DataFrame is', getsizeof(review_df), 'bytes')
print('hashed feature matrix nnz :', f.nnz)                 # non-zero entries
print('hashed feature matrix size:', f.data.nbytes, 'bytes')
# A full one-hot / bag-of-words matrix needs a column per distinct word
# (a vocabulary of tens of thousands of terms) AND that stored vocabulary;
# the hashed matrix is a fixed 2**18 wide with the vocabulary thrown away.

## Visualize the data & results

_Hash a list of many category strings into m buckets for m = 16, 64, 256. As the number of buckets m grows, how does the collision rate fall — the size-vs-accuracy trade-off?_

In [ ]:
import numpy as np

# 200 distinct category strings (e.g. user IDs / words). Real, deterministic.
cats = ['cat_%04d' % i for i in range(200)]

def collision_rate(categories, m):
    # Hash each category into one of m buckets (Python hash, made deterministic).
    buckets = np.array([(hash(c) & 0x7fffffff) % m for c in categories])
    # Count distinct categories per bucket; a bucket with 2+ is a collision.
    counts = np.bincount(buckets, minlength=m)
    collided = counts[buckets] > 1           # is this category's bucket shared?
    return collided.mean()

for m in [16, 32, 64, 128, 256]:
    print('m=%3d  collision rate = %.2f' % (m, collision_rate(cats, m)))
# -> m= 16  collision rate = 0.92
#    m= 32  collision rate = 0.78
#    m= 64  collision rate = 0.55
#    m=128  collision rate = 0.32
#    m=256  collision rate = 0.18
# More buckets -> fewer collisions, at the cost of a wider feature vector.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** You have an ad-tech model with about 50 million distinct user IDs as a categorical feature, and new IDs appear every hour. Why is one-hot encoding a non-starter, and how does feature hashing fix it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Count what one-hot would cost: one column per distinct ID plus a dictionary mapping each ID to its column. — _With 50 million IDs the vector is 50 million wide and the stored vocabulary is enormous — and it must grow every hour as new IDs arrive._
- Replace the lookup table with a hash function $h$ and a fixed bucket count $m$. — _Each ID's bucket is computed as $h(\text{id})\bmod m$ on demand, so there is nothing to store and the width is pinned to $m$._
- Note that a brand-new ID just hashes like any other. — _The transform is stateless, so streaming IDs need no vocabulary update — ideal for online learning._

**Answer:** One-hot needs a column per ID and a giant, ever-growing vocabulary — 50 million columns plus a dictionary that must be updated hourly. Feature hashing fixes the width at a chosen $m$ and stores no table: each ID's bucket is $h(\text{id})\bmod m$, recomputed on demand, so new streaming IDs are handled instantly. The cost is occasional collisions, kept small by picking $m$ large.

</details>

**Problem 2.** Two distinct words hash to the same bucket. With plain (unsigned) counting their contributions add up, biasing the bucket. How does the signed-hash trick reduce this bias?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Write the unsigned bucket: both words add $+1$, so the bucket holds the sum of their counts. — _Their signals are merged and the bucket is systematically inflated whenever both occur._
- Attach an independent sign $\xi(c)\in\{-1,+1\}$ to each word before adding. — _Now one word may add and the other subtract, so a colliding pair can cancel instead of always piling up._
- Look at the cross term in a dot product: it is $\xi(a)\xi(b)$, equally likely $+1$ or $-1$. — _Its expected value is zero, so the hashed dot product is an unbiased estimate of the true one._

**Answer:** With the signed hash, each category carries a random sign $\xi(c)\in\{-1,+1\}$. Colliding categories then add with opposite signs about half the time, so their contributions cancel in expectation rather than accumulate. The cross term $\xi(a)\xi(b)$ has mean zero, making the hashed inner product an unbiased estimate of the true inner product despite collisions.

</details>

**Problem 3.** A teammate hashed features with $m=2^{14}$ during training but, to save space at serving time, re-hashed with $m=2^{12}$. The model's accuracy collapsed. What went wrong?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall that a category's bucket is $h(c)\bmod m$ — it depends on $m$. — _Changing $m$ changes every category's bucket, so the serving features point at different columns than training did._
- Recognize the model's learned weights are tied to the training bucket assignment. — _Weight in column $j$ means nothing if column $j$ now holds different categories._
- Keep $h$ and $m$ (and any sign hash) identical at train and serve time. — _The hash function IS the feature definition; it must match everywhere for the model to work._

**Answer:** The bucket is $h(c)\bmod m$, so changing $m$ from $2^{14}$ to $2^{12}$ remapped every category to a different bucket. The model's weights were learned against the training buckets, so at serving time each weight now applies to the wrong features. The fix: use the exact same hash and $m$ (and sign hash) at train and serve time — the hash is part of the feature definition.

</details>